Install Dependencies

In [1]:
!pip install -q yt-dlp openai-whisper
!apt-get -qq install ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 20.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.8 MB/s eta 0:00:00


In [2]:
!ffmpeg -version

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-l

Download Video

In [3]:
from yt_dlp import YoutubeDL            # Used for downloading videos from YouTube.
url = "https://www.youtube.com/watch?v=zCYQEHBfYuU&t=110s"        # URL of the YouTube video to be downloaded

ydl_opts = {
    'format': 'mp4',                             # Download the video in MP4 format
    'outtmpl': '/content/video.%(ext)s'
}

with YoutubeDL(ydl_opts) as ydl:
    ydl.download([url])                         # Download the video from the provided URL

[youtube] Extracting URL: https://www.youtube.com/watch?v=zCYQEHBfYuU&t=110s
[youtube] zCYQEHBfYuU: Downloading webpage


[youtube] zCYQEHBfYuU: Downloading android vr player API JSON
[info] zCYQEHBfYuU: Downloading 1 format(s): 18
[download] Destination: /content/video.mp4
[download] 100% of   63.38MiB in 00:00:05 at 11.15MiB/s  


Extract Audio

In [6]:
# Use FFmpeg to extract audio from the downloaded video file
!ffmpeg -i /content/video.mp4 \
    -ar 16000 \
    -ac 1 \
    /content/audio.wav \
    -y

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

Load Whisper ( to convert audio to text )

In [7]:
import whisper

model = whisper.load_model("base")

100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 233MiB/s]


In [8]:
import torch
print(torch.cuda.is_available())

True


Generate Transcript

In [9]:
# Transcribe the audio file using the loaded Whisper model
result = model.transcribe(
    "/content/audio.wav",
    word_timestamps=True               # create timestamps
)

In [10]:
# Loop through the first 10 transcription segments
for seg in result["segments"][:10]:
    print(
        f"[{seg['start']:.2f} - {seg['end']:.2f}]",
        seg['text']
    )

[0.00 - 10.18]  It's time now to dive into our next session and it gives me immense delight to invite and introduce our moderator for this session.
[10.54 - 14.16]  Ladies and gentlemen, it's none other than Sonia Shanoi.
[14.30 - 22.72]  A leading voice in Indian business journalism, Sonia was assistant executive editor at CNBC TV 18 for over 16 years.
[22.72 - 35.72]  A finance and FinTech expert, today she's an independent content creator and the host of top podcasts like the Money Mindset, founder F-Ops and visionaries of India.
[36.44 - 42.02]  Please join me in welcoming Sonia Shanoi, ladies and gentlemen, a big round of applause.
[43.62 - 48.96]  Thank you to each and every one of you for being here and it is such an honour to be a part of this evening.
[48.96 - 52.78]  Because this is a topic that everyone is grappling with today.
[53.00 - 59.06]  So without wasting any time, let me invite someone who needs no introduction but I'm going to do it anyway.
[59.90 - 67.78]  Nanda N

Save as JSON FILE

In [11]:
 # Save the transcript data in JSON format
import json

transcript2 = []

for seg in result["segments"]:
    transcript2.append({
        "start": seg["start"],
        "end": seg["end"],
        "text": seg["text"]
    })

with open("/content/transcript2.json", "w") as f:
    json.dump(transcript2, f, indent=4)

Download JSON FILE

In [12]:
from google.colab import files

files.download('/content/transcript2.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>